In [1]:
!pip install torch-geometric
!pip install lifelines
!pip install pycox

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.7/115.7 kB 6.0 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=041efb91fb697c2262335496f707b2977bbba32257def3c4af892a7899c6d65d
  Stored in directory: /root/.cache/pip/wheels/8b/67/f4/2caaae2146198dcb824f31a303833b07b14a5ec863fb3acd7b
Successfully built autograd-gamma
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 11.9 M

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from lifelines.utils import concordance_index
from pycox.models.loss import CoxPHLoss

In [7]:
train_file_path = "1yVTNHjwM4hUbv0alTfW9hEn1itOVbHLg"
test_file_path = "1-1NGzpvsdLcBPFZU_aNOZYn-1BIMoWO2"
train_url = f"https://drive.google.com/uc?export=download&id={train_file_path}"
test_url = f"https://drive.google.com/uc?export=download&id={test_file_path}"
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

Preprocess Data

In [8]:
from sklearn.model_selection import KFold

# Define columns
event_col = "efs"
time_col = "efs_time"
feature_cols = [col for col in train_df.columns if col not in [event_col, time_col]]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
c_indices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"Fold {fold+1}:")

    # Split train and validation data for this fold
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()

    # Compute correlation matrix for this fold
    correlation_matrix = train_fold[feature_cols].corr()

    # Remove highly correlated features (threshold = 0.85)
    correlation_threshold = 0.85
    correlated_features = set()

    for i in range(len(correlation_matrix.columns)):
        for j in range(i):
            if abs(correlation_matrix.iloc[i, j]) > correlation_threshold:
                correlated_features.add(correlation_matrix.columns[i])  # Keep only one from each correlated pair

    print(f"Removing {len(correlated_features)} highly correlated features for Fold {fold+1}:", correlated_features)

    # Drop correlated features from train and validation folds
    train_fold.drop(columns=correlated_features, inplace=True)
    val_fold.drop(columns=correlated_features, inplace=True)

    # Update feature columns dynamically per fold
    fold_feature_cols = [col for col in feature_cols if col not in correlated_features]

    # Standardize features per fold
    scaler = StandardScaler()
    train_fold[fold_feature_cols] = scaler.fit_transform(train_fold[fold_feature_cols])
    val_fold[fold_feature_cols] = scaler.transform(val_fold[fold_feature_cols])

    # Convert to tensors
    train_x = torch.tensor(train_fold[fold_feature_cols].values, dtype=torch.float32)
    val_x = torch.tensor(val_fold[fold_feature_cols].values, dtype=torch.float32)
    train_durations = torch.tensor(train_fold[time_col].values, dtype=torch.float32)
    train_events = torch.tensor(train_fold[event_col].values, dtype=torch.float32)
    val_durations = torch.tensor(val_fold[time_col].values, dtype=torch.float32)
    val_events = torch.tensor(val_fold[event_col].values, dtype=torch.float32)

    # Compute class weights per fold
    event_counts = train_events.sum().item()  # Number of events (1s)
    censored_counts = len(train_events) - event_counts  # Number of censored cases (0s)

    # Check class distribution
    print(f"Fold {fold+1} - Events (1s): {event_counts}, Censored (0s): {censored_counts}, Ratio: {event_counts / censored_counts:.4f}")

    # Assign higher weight to the minority class
    weight_for_event = (censored_counts / (event_counts + 1e-5)) ** 0.5  # Adjust weighting to reduce imbalance impact
    weight_for_censored = 1.0  # Reference weight

    # Create sample weights
    sample_weights = torch.where(train_events == 1, weight_for_event, weight_for_censored)

    # Store validation labels for later evaluation
    val_durations = torch.tensor(val_fold[time_col].values, dtype=torch.float32)
    val_events = torch.tensor(val_fold[event_col].values, dtype=torch.float32)

    print(f"Fold {fold+1} complete!\n")

print("K-Fold Data Processing Complete!")

Fold 1:
Removing 13 highly correlated features for Fold 1: {'hla_high_res_8', 'prod_type_PB', 'hla_match_a_high', 'hla_match_b_low', 'prod_type_BM', 'tce_div_match_Unknown', 'hla_high_res_6', 'ethnicity_Not Hispanic or Latino', 'donor_related_Unrelated', 'melphalan_dose_N/A, Mel not given', 'hla_low_res_8', 'hla_match_b_high', 'hla_low_res_10'}
Fold 1 - Events (1s): 12455.0, Censored (0s): 10585.0, Ratio: 1.1767
Fold 1 complete!

Fold 2:
Removing 13 highly correlated features for Fold 2: {'hla_high_res_8', 'prod_type_PB', 'hla_match_a_high', 'hla_match_b_low', 'prod_type_BM', 'tce_div_match_Unknown', 'hla_high_res_6', 'ethnicity_Not Hispanic or Latino', 'donor_related_Unrelated', 'melphalan_dose_N/A, Mel not given', 'hla_low_res_8', 'hla_match_b_high', 'hla_low_res_10'}
Fold 2 - Events (1s): 12349.0, Censored (0s): 10691.0, Ratio: 1.1551
Fold 2 complete!

Fold 3:
Removing 13 highly correlated features for Fold 3: {'hla_high_res_8', 'prod_type_PB', 'hla_match_a_high', 'hla_match_b_low',

Construct the graph

In [9]:
import torch
from torch_geometric.data import Data
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def construct_graph(data, threshold=0.8, max_edges_per_node=10):
    """
    Constructs a graph using cosine similarity.
    - Connects nodes with similarity > threshold.
    - Limits the number of edges per node to avoid over-connection.
    """
    similarity_matrix = cosine_similarity(data.numpy())
    edge_index = []

    for i in range(len(similarity_matrix)):
        # Get indices of nodes with similarity > threshold (excluding itself)
        similar_nodes = np.where(similarity_matrix[i] > threshold)[0]
        similar_nodes = similar_nodes[similar_nodes != i]  # Remove self-loops

        # Limit the number of connections per node
        if len(similar_nodes) > max_edges_per_node:
            similar_nodes = np.argsort(similarity_matrix[i][similar_nodes])[-max_edges_per_node:]

        # Add edges
        for j in similar_nodes:
            edge_index.append([i, j])
            edge_index.append([j, i])  # Ensure bidirectional edges

    # Convert to tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return edge_index

# Construct graph using the improved function
graph_edges = construct_graph(train_x, threshold=0.8, max_edges_per_node=10)

# Ensure edge indices are valid
if graph_edges.max() >= train_x.shape[0]:
    raise ValueError(f"Graph contains out-of-bounds indices! Max index: {graph_edges.max()}, Valid range: 0-{train_x.shape[0]-1}")

# Create graph data object
graph_data = Data(x=train_x, edge_index=graph_edges)

Define the model

In [11]:
class GNNModel(nn.Module):
    def __init__(self, input_dim):
        super(GNNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, 128)
        self.conv2 = GCNConv(128, 64)
        self.fc = nn.Linear(64, 1)

    def forward(self, x, edge_index):
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        x = self.fc(x)
        return x.squeeze()

def get_gnn_model(input_dim, lr=0.005):
    """Initialize a new GNN model and optimizer for each fold."""
    model = GNNModel(input_dim=input_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = CoxPHLoss()
    return model, optimizer, loss_fn

def train_gnn_for_fold(model, optimizer, loss_fn, train_x, train_durations, train_events, graph_edges, sample_weights, epochs=500):
    """Train GNN for one K-Fold split."""
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # Forward pass
        risk_scores = model(train_x, graph_edges)
        loss = loss_fn(risk_scores, train_durations, train_events)

        # Apply sample weights to loss
        weighted_loss = (loss * sample_weights).mean()

        # Backpropagation
        weighted_loss.backward()
        optimizer.step()

        # Logging every 10 epochs
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Weighted Loss = {weighted_loss.item():.4f}")

    return model  # Return trained model for this fold

Training

In [12]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from lifelines.utils import concordance_index
from sklearn.model_selection import KFold
import numpy as np

# Define K-Fold settings
kf = KFold(n_splits=5, shuffle=True, random_state=42)
c_indices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"Fold {fold+1}:")

    # Split train and validation data for this fold
    train_fold = train_df.iloc[train_idx].copy()
    val_fold = train_df.iloc[val_idx].copy()

    # Compute correlation matrix and remove highly correlated features for this fold
    correlation_matrix = train_fold[feature_cols].corr()
    correlation_threshold = 0.85
    correlated_features = set()

    for i in range(len(correlation_matrix.columns)):
        for j in range(i):
            if abs(correlation_matrix.iloc[i, j]) > correlation_threshold:
                correlated_features.add(correlation_matrix.columns[i])  # Keep only one from each correlated pair

    print(f"Removing {len(correlated_features)} highly correlated features for Fold {fold+1}: {correlated_features}")

    train_fold.drop(columns=correlated_features, inplace=True)
    val_fold.drop(columns=correlated_features, inplace=True)

    fold_feature_cols = [col for col in feature_cols if col not in correlated_features]

    # Standardize features per fold
    scaler = StandardScaler()
    train_fold[fold_feature_cols] = scaler.fit_transform(train_fold[fold_feature_cols])
    val_fold[fold_feature_cols] = scaler.transform(val_fold[fold_feature_cols])

    # Convert to tensors
    train_x = torch.tensor(train_fold[fold_feature_cols].values, dtype=torch.float32)
    val_x = torch.tensor(val_fold[fold_feature_cols].values, dtype=torch.float32)
    train_durations = torch.tensor(train_fold[time_col].values, dtype=torch.float32)
    train_events = torch.tensor(train_fold[event_col].values, dtype=torch.float32)
    val_durations = torch.tensor(val_fold[time_col].values, dtype=torch.float32)
    val_events = torch.tensor(val_fold[event_col].values, dtype=torch.float32)

    # Compute class weights per fold
    event_counts = train_events.sum().item()
    censored_counts = len(train_events) - event_counts
    print(f"Fold {fold+1} - Events (1s): {event_counts}, Censored (0s): {censored_counts}, Ratio: {event_counts / censored_counts:.4f}")

    weight_for_event = (censored_counts / (event_counts + 1e-5)) ** 0.5
    weight_for_censored = 1.0
    sample_weights = torch.where(train_events == 1, weight_for_event, weight_for_censored)

    # Construct graph for training set
    graph_edges = construct_graph(train_x, threshold=0.8, max_edges_per_node=10)
    graph_data = Data(x=train_x, edge_index=graph_edges)

    # Initialize a new model for each fold
    model, optimizer, loss_fn = get_gnn_model(input_dim=train_x.shape[1])

    # Train the model for this fold
    for epoch in range(500):
        model.train()
        optimizer.zero_grad()
        risk_scores = model(graph_data.x, graph_data.edge_index)
        loss = loss_fn(risk_scores, train_durations, train_events)
        weighted_loss = (loss * sample_weights).mean()
        weighted_loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Fold {fold+1}, Epoch {epoch}: Weighted Loss = {weighted_loss.item():.4f}")

    # Construct graph for validation set
    val_graph_edges = construct_graph(val_x, threshold=0.8, max_edges_per_node=10)

    # Ensure edge indices are valid for validation
    if val_graph_edges.max() >= val_x.shape[0]:
        raise ValueError(f"Validation graph contains out-of-bounds indices! Max index: {val_graph_edges.max()}, Valid range: 0-{val_x.shape[0]-1}")

    # Evaluate model on this fold
    model.eval()
    val_risk_scores = model(val_x, val_graph_edges)
    c_index = concordance_index(val_durations.numpy(), -val_risk_scores.detach().numpy(), val_events.numpy())
    c_indices.append(c_index)

    print(f"Fold {fold+1} C-Index: {c_index:.4f}\n")

# Compute mean C-Index across folds
mean_c_index = np.mean(c_indices)
print(f"Mean C-Index across folds: {mean_c_index:.4f}")

Fold 1:
Removing 13 highly correlated features for Fold 1: {'hla_high_res_8', 'prod_type_PB', 'hla_match_a_high', 'hla_match_b_low', 'prod_type_BM', 'tce_div_match_Unknown', 'hla_high_res_6', 'ethnicity_Not Hispanic or Latino', 'donor_related_Unrelated', 'melphalan_dose_N/A, Mel not given', 'hla_low_res_8', 'hla_match_b_high', 'hla_low_res_10'}
Fold 1 - Events (1s): 12455.0, Censored (0s): 10585.0, Ratio: 1.1767
Fold 1, Epoch 0: Weighted Loss = 9.3415
Fold 1, Epoch 10: Weighted Loss = 9.1362
Fold 1, Epoch 20: Weighted Loss = 9.0824
Fold 1, Epoch 30: Weighted Loss = 9.0073
Fold 1, Epoch 40: Weighted Loss = 8.9249
Fold 1, Epoch 50: Weighted Loss = 8.8361
Fold 1, Epoch 60: Weighted Loss = 8.7675
Fold 1, Epoch 70: Weighted Loss = 8.7037
Fold 1, Epoch 80: Weighted Loss = 8.6609
Fold 1, Epoch 90: Weighted Loss = 8.6631
Fold 1, Epoch 100: Weighted Loss = 8.5934
Fold 1, Epoch 110: Weighted Loss = 8.5394
Fold 1, Epoch 120: Weighted Loss = 8.5067
Fold 1, Epoch 130: Weighted Loss = 8.4823
Fold 1,